# 1.获取大模型

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
import dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  # 加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('OPENAI_BASE_URL')

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")  # 默认使用 gpt-3.5-turbo

# 直接提供问题，并调用 LLM
response = llm.invoke("什么是大模型？")
print(response)


# 2.使用提示次模板

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 使用提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")
])

# 创建链
chain = prompt | llm

# 调用链
response = chain.invoke({"input": "人工智能"})
print(response)


# 3.使用输出解析器

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
import dotenv
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  # 加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('OPENAI_BASE_URL')

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")  # 默认使用 gpt-3.5-turbo


# 使用提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")
])

# 创建outputparser
output_parser = JsonOutputParser()


# 创建链
chain = prompt | llm |output_parser


# 调用链
response = chain.invoke({"input": "人工智能"})
print(response)


4.使用向量存储

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4

# 使用 WebBaseLoader 加载网页内容
loader = WebBaseLoader(
    web_path="https://www.gov.cn/xinwen/2020-06/01/content_5516649.htm",
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
)
docs = loader.load()
# print(docs)

# 对于嵌入模型，这里通过 API 调用
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(len(documents))

# 向量存储 embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)


# 5.RAG检索增强生成